In [1]:
library(dplyr)
library(stringr)


Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



In [2]:
feats <- read.csv("/home/lz80/rdf/sp161/shared/soccer-decision-making-r/sportec/xgb_features.csv") |> rename(
  i_angle = `i_angle....intended_angle`,
  i_x_velo = `i_x_velo....intended_x_velo`,
  i_y_velo = `i_y_velo....intended_y_velo`,
  EVENT_ID = event_id,
  MUID = match_id
) |> mutate(
  MUID = str_sub(MUID, 1, -4 - 1)
)
labels <- read.csv("/home/lz80/rdf/sp161/shared/soccer-decision-making-r/sportec/xgb_labels.csv") 

In [5]:
build_dataset <- function(feats, labels, offensive = TRUE, complete = TRUE) {
  if(offensive){
    labels <- labels |> select(EVENT_ID, MUID, offense_xG)
  } else {
    labels <- labels |> select(EVENT_ID, MUID, defense_xG)
  }
  feats |> inner_join(labels, by = c("EVENT_ID", "MUID"))
  
}
passes = build_dataset(feats, labels)

In [7]:
passes |> dim()

[1] 253843     66

In [76]:
prepare_training_matrix <- function(passes, offensive = TRUE) {
  label_col <- if (offensive) "offense_xG" else "defense_xG"

  feature_cols <- setdiff(colnames(passes), c("MUID", "EVENT_ID", "X", label_col))

  model_df <- passes |>
    dplyr::select(dplyr::all_of(c(feature_cols, label_col))) |>
    dplyr::filter(!is.na(.data[[label_col]]))   # use chosen label

  mf <- model.frame(
    reformulate(feature_cols, response = label_col),
    data = model_df,
    na.action = na.pass
  )

  X <- model.matrix(
    reformulate(feature_cols, response = NULL, intercept = FALSE),
    data = mf
  )


  list(
    X = X,
    y = model_df[[label_col]],
    feature_cols = colnames(X)
  )
}

training_mat <- prepare_training_matrix(passes)

In [79]:
training_mat$X |> colnames() |> length()
#training_mat$y

[1] 62